# Build a workspace through the API

Create a workspace, add tabs, lay modules out on the grid, and render a plot.

Edit the **Configuration** cell, then run top to bottom.

The visual side of the app is `Workspace -> Tab -> Module`. A workspace is purely a container: it
does not own or store data. Modules reference existing uploads and datasets by id, so there is no
such thing as uploading *into* a workspace.

### Before you start

```bash
pip install "git+https://github.com/MassDynamics/md-python.git" python-dotenv
```

with credentials in a `.env` file next to this notebook:

```
MD_API_BASE_URL=https://app.massdynamics.com/api
MD_AUTH_TOKEN=<your token, no "Bearer " prefix>
```

## Setup

In [ ]:
import json
import os
from collections import defaultdict

from dotenv import load_dotenv
from md_python import MDClient

load_dotenv()
assert os.getenv("MD_AUTH_TOKEN"), "MD_AUTH_TOKEN is not set (check your .env)"

client = MDClient(version="v2")
print("API:", client.base_url)
print("Health:", client.health.check())

## Configuration

The only cell you need to edit.

In [ ]:
WORKSPACE_NAME = "YOUR WORKSPACE NAME"
WORKSPACE_DESCRIPTION = "Built through the API"

TAB_NAMES = ["Overview", "Differential abundance"]

# Part of a dataset name to search for in section 2.
DATASET_SEARCH = "pairwise"

# A PAIRWISE dataset to plot, and which comparison to show.
# Leave as None to build the layout without a data module.
PAIRWISE_DATASET_ID = None
COMPARISON_LEFT = "TREATED_GROUP"
COMPARISON_RIGHT = "CONTROL_GROUP"

## 1. Create the workspace and its tabs

Both take a name and little else.

In [ ]:
workspace = client.workspaces.create(
    name=WORKSPACE_NAME,
    description=WORKSPACE_DESCRIPTION,
)
workspace_id = str(workspace.id)
print("Workspace:", workspace_id)

tab_ids = {}
for name in TAB_NAMES:
    tab = client.workspaces.tabs.create(workspace_id=workspace_id, name=name)
    tab_ids[name] = str(tab.id)
    print(f"  tab {name!r}: {tab.id}")

tab_id = tab_ids[TAB_NAMES[0]]

## 2. Find the data you want to plot

Modules reference datasets by id, so before you can place a plot you need to find the right dataset.

`client.datasets.query()` searches by **partial name** and filters by type and state. It returns
`{"data": [...], "pagination": {...}}`, 50 per page.

```python
client.datasets.query(search="my experiment")            # name contains
client.datasets.query(type=["PAIRWISE"], state=["COMPLETED"])
client.datasets.query(upload_id=...)                     # everything from one upload
```

Valid `type` values are `DEMO`, `DOSE_RESPONSE`, `DOSE_RESPONSE_AGGREGATE`, `ENRICHMENT`,
`IMPUTATION`, `INTENSITY`, `NORMALISATION_AND_IMPUTATION` and `PAIRWISE`. Anything else is a 400.
Note `ANOVA` datasets exist but cannot be filtered for, so find those with `search=` instead.

In [ ]:
result = client.datasets.query(search=DATASET_SEARCH, page=1)
found = result["data"]
print(f"{result['pagination']['total_count']} datasets matching {DATASET_SEARCH!r}"
      f" (showing {len(found)})\n")

for d in found[:15]:
    print(f"  {str(d.get('type')):<18} {str(d.get('state')):<10} "
          f"{d.get('id')}  {d.get('name')}")

### Narrow it to what you can actually plot

A volcano needs a `PAIRWISE` dataset that has finished. Filtering on both saves you from picking
one that is still processing.

In [ ]:
pairwise_datasets = client.datasets.query(
    type=["PAIRWISE"], state=["COMPLETED"], page=1
)["data"]

print(f"{len(pairwise_datasets)} completed PAIRWISE datasets on this page:\n")
for d in pairwise_datasets[:10]:
    print(f"  {d['id']}  {d['name']}")

### What does one of them look like?

`get_by_id()` returns the full record. Three parts are worth reading:

| | |
|---|---|
| `tables` | what you can download. A PAIRWISE dataset carries `output_comparisons`. |
| `input_dataset_ids` | what it was computed from, so you can walk back to the intensity dataset |
| `job_run_params` | the settings it was run with |

For a pairwise dataset `job_run_params` is especially useful: `condition_comparisons` lists every
contrast the dataset actually contains, and `condition_column` names the metadata column they were
grouped on. Those are exactly the values the volcano module needs later, so read them from here
rather than guessing.

In [ ]:
if pairwise_datasets:
    inspect_id = pairwise_datasets[0]["id"]
    ds = client.datasets.get_by_id(inspect_id)

    print(f"name   : {ds.name}")
    print(f"id     : {ds.id}")
    print(f"type   : {ds.type}   state: {ds.state}")
    print(f"tables : {[t.name for t in (ds.tables or [])]}")
    print(f"inputs : {[str(i) for i in ds.input_dataset_ids]}")

    params = ds.job_run_params or {}
    print(f"\ncondition_column: {params.get('condition_column')}")
    pairs = (params.get("condition_comparisons") or {}).get(
        "condition_comparison_pairs", []
    )
    print(f"{len(pairs)} comparison(s) in this dataset:")
    for left, right in pairs[:8]:
        print(f"    {left}  vs  {right}")

    if pairs:
        left, right = pairs[0]
        print("\nTo plot the first one, put these in the Configuration cell:")
        print(f'    PAIRWISE_DATASET_ID = "{ds.id}"')
        print(f'    COMPARISON_LEFT     = "{left}"')
        print(f'    COMPARISON_RIGHT    = "{right}"')
else:
    print("No completed PAIRWISE datasets found. Run a pairwise comparison first.")

### Reading the numbers

To pull the results themselves, get a presigned URL for one of the dataset's tables:

```python
url = client.datasets.download_table_url(dataset_id, "output_comparisons", format="csv")

import pandas as pd
df = pd.read_csv(url)          # one row per entity, one column trio per comparison
```

`output_comparisons` has `AdjPValue <left> - <right>`, `PValue ...` and `MaxLog2FoldChange ...`
columns per comparison, plus a bare aggregate `AdjPValue`. Dose-response datasets expose
`output_curves` and `output_volcanoes` instead, where the half-maximal dose column is `ED50`.

## 3. Which modules can I place?

`client.module_registry.list()` returns every module the app can render. The `<- needs no settings`
marker flags the handful you can drop onto a tab with no configuration at all, which is useful when
you want to build the page shape first and wire up data later.

In [ ]:
modules = client.module_registry.list()

by_group = defaultdict(list)
for m in modules:
    by_group[str(m.group)].append(m)

print(f"{len(modules)} modules\n")
for group in sorted(by_group):
    print(group)
    for m in sorted(by_group[group], key=lambda x: str(x.id)):
        no_config = not m.missing_required_keys(m.defaults())
        print(f"    {str(m.id):<40} {m.name}{'  <- needs no settings' if no_config else ''}")
    print()

## 4. What parameters does a module take?

`client.module_registry.get(item_id)` returns the module's `input_settings`: field name, type,
default, and whether it is required. This is the source of truth. The server rejects any settings
key the module does not declare.

**Requirements cascade.** A field carrying a `when` clause only becomes required once the field it
depends on is set, so the "you must supply" list grows as you fill it in. Fill in what it asks for,
then run this again.

In [ ]:
def show_module(item_id):
    """Print one module's parameters. Fetches its own registry entry, so this
    cell works on its own."""
    m = client.module_registry.get(item_id)
    assert m, f"{item_id!r} is not in the module registry"

    schema = m.input_settings or {}
    items = list(schema.items()) if isinstance(schema, dict) else [
        (s.get("key"), s) for s in schema if isinstance(s, dict)
    ]
    print(f"{m.id}  ({m.group})  -  {len(items)} settings\n")
    for key, spec in items:
        if not isinstance(spec, dict):
            print(f"  {key}")
            continue
        bits = []
        if spec.get("fieldType"):
            bits.append(str(spec["fieldType"]))
        if spec.get("default") is not None:
            bits.append(f"default={spec['default']!r}")
        if spec.get("required") is True or any(
            r.get("name") == "is_required" for r in (spec.get("rules") or [])
        ):
            bits.append("REQUIRED")
        if spec.get("when"):
            bits.append(f"when={spec['when']}")
        print(f"  {str(key):<34} {'  '.join(bits)}")

    missing = m.missing_required_keys(m.defaults())
    print()
    print(f"You must supply: {missing}" if missing
          else "Nothing required. Can be placed with settings={}.")

show_module("pairwise_volcano_plot")

## 5. The grid

A tab is a [react-grid-layout](https://github.com/jbaysolutions/vue-grid-layout) with **12 columns**
(`DashboardTabModules.vue`, `:col-num="12"`). Every module is placed with four integers:

| | |
|---|---|
| `x` | column it starts at, `0` to `11` |
| `y` | row it starts at, `0` upwards |
| `width` | how many of the 12 columns it spans |
| `height` | how many rows tall |

Width is the number that carries the meaning. 12 is full width, so:

```
width 12  ->  one per row
width  6  ->  two side by side    (x = 0, 6)
width  4  ->  three side by side  (x = 0, 4, 8)
width  3  ->  four side by side   (x = 0, 3, 6, 9)
```

`x + width` must not exceed 12 or the module wraps. Height is in grid rows, not pixels (a row is
roughly 30px), so a plot usually wants 8 to 12. Step `y` down by the height of the row above or
modules overlap.

```
y=0   [ heading, width 12                                  ]
y=2   [ panel w=4     ][ panel w=4     ][ panel w=4        ]
y=8   [ half w=6              ][ half w=6                  ]
y=14  [ volcano, width 12                                  ]
```

### Placing them

`create_with_defaults()` fetches the registry entry, fills in every declared default, merges your
settings on top, and fails fast if a required setting has no default and you did not supply it.

Prefer it over `create()`: the API does not merge defaults server-side, so a partial settings hash
renders as a broken widget in the app.

### The `text` module takes plain text

Write the words you want to see and nothing else. **Workspaces do not allow HTML tags**, and
markdown is not rendered either, so `"## Heading"` shows up literally. When you want a heading, use
the `heading` module, which exists for exactly that.

Worth knowing because it will mislead you: the registry still describes the field as *"HTML and
embedded base64 imgs"* and the app renders it with `v-html`. Send HTML on the strength of that and
it will store without complaint. Workspaces are plain text by policy, so send plain text.

There is also **no settings button** on a text module in the app, by design. Its instruction
declares `INPUT_PROPERTIES = {}`, so there is no settings form to render. You edit it inline
instead: click into the module and a WYSIWYG toolbar appears in place. The `text` key exists in the
registry manifest purely so the API can set the body, which is what we do here.


In [ ]:
def place(item_id, x, y, width, height, settings=None):
    assert x + width <= 12, f"x({x}) + width({width}) exceeds the 12-column grid"
    m = client.workspaces.modules.create_with_defaults(
        workspace_id=workspace_id, tab_id=tab_id, item_id=item_id,
        x=x, y=y, width=width, height=height, settings=settings or {},
    )
    print(f"  {item_id:<26} x={x:<3} y={y:<3} w={width:<3} h={height:<3} -> {m.id}")
    return m


print("row 1: full width")
place("text", 0, 0, 12, 2, {"text": "## Results\nBuilt through the API."})

print("\nrow 2: three across, width 4")
for i, x in enumerate([0, 4, 8], start=1):
    place("text", x, 2, 4, 6, {"text": f"Panel {i}\n\nx={x}, width=4"})

print("\nrow 3: two across, width 6")
for i, x in enumerate([0, 6], start=1):
    place("text", x, 8, 6, 6, {"text": f"Half {i}\n\nx={x}, width=6"})

## 6. A data module

Structured field types are not plain ids. Two you will meet immediately:

```python
DatasetsSelect       {"individualResults": [{"id": ..., "name": ...}]}   # id AND name
ConditionComparison  {"comparison": {"conditionPair": "<left> - <right>",
                                     "left": ..., "right": ...}}
```

`conditionPair` must be exactly `"{left} - {right}"`. Send a bare list of ids and the API answers
*"DatasetsSelect must be an object with individualResults {id,name}"*.

Source: `workflow/app/models/module_registry/structured_field_violations.rb`.

In [ ]:
if PAIRWISE_DATASET_ID:
    dataset = client.datasets.get_by_id(PAIRWISE_DATASET_ID)
    assert dataset, f"No dataset {PAIRWISE_DATASET_ID!r}"

    print("row 4: volcano, full width")
    volcano = place(
        "pairwise_volcano_plot", 0, 14, 12, 12,
        settings={
            "datasetsSearch": {
                "individualResults": [
                    {"id": PAIRWISE_DATASET_ID, "name": dataset.name}
                ]
            },
            "experimentAndConditionComparison": {
                "comparison": {
                    "conditionPair": f"{COMPARISON_LEFT} - {COMPARISON_RIGHT}",
                    "left": COMPARISON_LEFT,
                    "right": COMPARISON_RIGHT,
                }
            },
        },
    )
else:
    volcano = None
    print("Skipped: set PAIRWISE_DATASET_ID to place a volcano plot.")

### What is on the tab now

In [ ]:
for m in sorted(client.workspaces.modules.list(workspace_id, tab_id),
                key=lambda m: (m.y, m.x)):
    print(f"  {str(m.item_id):<26} x={m.x:<3} y={m.y:<3} w={m.width:<3} h={m.height}")

## Where to go next

The workspace is now live in the app. To keep working with it:

```python
client.workspaces.list_all()                              # find it again
client.workspaces.tabs.list_all(workspace_id)             # its tabs
client.workspaces.modules.list(workspace_id, tab_id)      # its modules
client.workspaces.modules.update(...)                     # move or resize
client.workspaces.modules.delete(workspace_id, tab_id, module_id)
```

`client.workspaces.modules.create_text(...)` is a shortcut for text modules if you are writing a lot
of narrative.

If a module is rejected, re-run `show_module("<item_id>")` and compare the field list against what
you are sending. The registry is the source of truth and it does change.